# Explainable AI for Customer Churn Prediction

**Dissertation title:** *Explainable Artificial Intelligence for Customer Churn Prediction: A Data-Driven and Ethical Machine Learning Study*

This notebook provides a complete MSc-level workflow for:
1. Data loading and quality checks
2. Exploratory data analysis (EDA)
3. Data preprocessing and train-test split
4. Predictive modelling (Logistic Regression, Random Forest, SVM)
5. Model evaluation with classification metrics and ROC analysis
6. Explainable AI (feature importance + SHAP)
7. Advanced model extensions (Gradient Boosting family)
8. Dissertation-ready outputs (saved figures/tables)

---

## How to use this notebook
- Place your CSV file in the project root or update `DATA_PATH` below.
- Run cells in order from top to bottom.
- All dissertation outputs are saved in `outputs/figures/` and `outputs/tables/`.


In [ ]:
# =============================
# 1. IMPORTS AND CONFIGURATION
# =============================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
)

# Optional advanced model (XGBoost) if installed
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

import shap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Paths
BASE_DIR = Path(".").resolve()
DATA_PATH = BASE_DIR / "churn.csv"  # Change this if your file has a different name/path
FIG_DIR = BASE_DIR / "outputs" / "figures"
TABLE_DIR = BASE_DIR / "outputs" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Base directory:", BASE_DIR)
print("Data path:", DATA_PATH)
print("Figures will be saved to:", FIG_DIR)
print("Tables will be saved to:", TABLE_DIR)
print("XGBoost available:", XGBOOST_AVAILABLE)


## Section A: Data Loading

### What this section does
- Reads the synthetic churn dataset from a CSV file.
- Performs basic checks (shape, sample rows, column names, and data types).

### Why this is important
- Establishes a reliable starting point for analysis.
- Identifies issues early (missing values, incorrect data types, naming inconsistencies).

### Dissertation writing guidance
- **Methodology:** Describe data source, synthetic/anonymised nature, and variable overview.
- **Results:** Briefly report dataset size and key structural observations.


In [ ]:
# ==================
# 2. LOAD THE DATA
# ==================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Please place your CSV there or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

df.head()


## Section B: Exploratory Data Analysis (EDA)

### What this section does
- Summarises descriptive statistics and missing values.
- Visualises target class balance (churn vs non-churn).
- Explores distributions and key relationships.

### Why this is important
- Helps understand data behaviour before modelling.
- Reveals potential imbalance and informative patterns.

### Dissertation writing guidance
- **Methodology:** Explain EDA steps and rationale for plotting choices.
- **Results:** Present main patterns with references to figures.


In [ ]:
# =====================================
# 3. BASIC QUALITY CHECKS + EDA TABLES
# =====================================

# Standardise column names for easier handling
# (keeps meaning intact but avoids spaces/odd characters issues)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Candidate target names (supports common churn datasets)
possible_targets = ["Exited", "Churn", "churn", "Target", "target"]
target_col = next((c for c in possible_targets if c in df.columns), None)

if target_col is None:
    raise ValueError(
        "Target column not found. Expected one of: "
        f"{possible_targets}. Found columns: {df.columns.tolist()}"
    )

print("Using target column:", target_col)

# Missing values summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary.to_frame(name="missing_count")
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100).round(2)

missing_summary.to_csv(TABLE_DIR / "table_missing_values.csv")

print("Saved missing value summary to table_missing_values.csv")
missing_summary.head(15)


In [ ]:
# =================================
# 4. EDA VISUALISATIONS (SAVE PLOTS)
# =================================

# 4.1 Target distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x=target_col, data=df, palette="Set2")
ax.set_title("Target Class Distribution")
ax.set_xlabel("Churn (1 = Yes, 0 = No)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_target_distribution.png", dpi=300)
plt.show()

# 4.2 Numeric feature distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

# Keep core numeric columns often present in churn datasets (if available)
preferred_numeric = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"
]
plot_numeric = [c for c in preferred_numeric if c in numeric_cols]

if len(plot_numeric) == 0:
    plot_numeric = numeric_cols[:6]  # fallback

fig, axes = plt.subplots(len(plot_numeric), 1, figsize=(8, 3 * len(plot_numeric)))
if len(plot_numeric) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_numeric):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_numeric_distributions.png", dpi=300)
plt.show()

# 4.3 Correlation heatmap (numeric only)
if len(numeric_cols) > 1:
    plt.figure(figsize=(10, 8))
    corr = df[numeric_cols + [target_col]].corr(numeric_only=True)
    sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap (Numeric Features + Target)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_correlation_heatmap.png", dpi=300)
    plt.show()


## Section C: Data Preprocessing and Train-Test Split

### What this section does
- Separates features (`X`) and target (`y`).
- Splits data into training and test sets.
- Builds preprocessing pipelines:
  - Numeric: imputation + standardisation
  - Categorical: imputation + one-hot encoding

### Why this is important
- Prevents data leakage by fitting preprocessing only on training data.
- Ensures fair and reproducible model evaluation.

### Dissertation writing guidance
- **Methodology:** Explain split ratio, stratification, and preprocessing choices.
- **Results:** Mention that identical preprocessing was used for model comparability.


In [ ]:
# ========================================
# 5. PREPROCESSING + TRAIN/TEST SPLIT
# ========================================

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# Remove row identifiers if present
id_like_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in X.columns]
if id_like_cols:
    X = X.drop(columns=id_like_cols)
    print("Dropped non-predictive identifier columns:", id_like_cols)

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


## Section D: Baseline Model Training

### What this section does
- Trains three widely used classifiers:
  1. Logistic Regression
  2. Random Forest
  3. Support Vector Machine (SVM)

### Why this is important
- Combines interpretable linear modelling (Logistic Regression) with non-linear methods.
- Supports robust comparative analysis.

### Dissertation writing guidance
- **Methodology:** Justify model selection and hyperparameter defaults.
- **Results:** Report comparative metric performance and identify best model.


In [ ]:
# ==========================
# 6. MODEL TRAINING
# ==========================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

trained_pipelines = {}

for name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    clf.fit(X_train, y_train)
    trained_pipelines[name] = clf
    print(f"Trained: {name}")


## Section E: Model Evaluation

### What this section does
- Calculates core classification metrics: accuracy, precision, recall, F1-score, ROC-AUC.
- Generates confusion matrices and ROC curves for model comparison.
- Saves results as tables and publication-quality figures.

### Why this is important
- Single metrics can be misleading in churn prediction (especially with class imbalance).
- Multiple metrics provide a balanced view of performance.

### Dissertation writing guidance
- **Methodology:** Define each metric and explain threshold-based classification.
- **Results:** Compare models using both numeric tables and visual evidence.


In [ ]:
# ==========================
# 7. MODEL EVALUATION
# ==========================

def evaluate_model(name, pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
    }

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    return metrics, cm, report, fpr, tpr

all_metrics = []
roc_data = {}
conf_matrices = {}

for model_name, model_pipeline in trained_pipelines.items():
    metrics, cm, report, fpr, tpr = evaluate_model(model_name, model_pipeline, X_test, y_test)
    all_metrics.append(metrics)
    roc_data[model_name] = (fpr, tpr, metrics["ROC_AUC"])
    conf_matrices[model_name] = cm

    print("\n" + "="*60)
    print(f"Classification report: {model_name}")
    print(report)

metrics_df = pd.DataFrame(all_metrics).sort_values(by="F1", ascending=False)
metrics_df.to_csv(TABLE_DIR / "table_model_metrics_baseline.csv", index=False)
metrics_df


In [ ]:
# ================================================
# 8. CONFUSION MATRICES + ROC CURVES (SAVE FIGURES)
# ================================================

# Confusion matrices
n_models = len(conf_matrices)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (model_name, cm) in zip(axes, conf_matrices.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix\n{model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_confusion_matrices_baseline.png", dpi=300)
plt.show()

# ROC curves
plt.figure(figsize=(8, 6))
for model_name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Baseline Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_roc_curves_baseline.png", dpi=300)
plt.show()


## Section F: Explainable AI (Feature Importance + SHAP)

### What this section does
- Uses model-based feature importance (Random Forest).
- Uses SHAP values for global feature impact and local explanations.

### Why this is important
- Improves transparency and interpretability of churn predictions.
- Supports ethical and accountable use of AI in business settings.

### Dissertation writing guidance
- **Methodology:** Describe SHAP conceptually (Shapley values from game theory).
- **Results:** Report top drivers of churn and discuss business/ethical implications.


In [ ]:
# =================================
# 9. FEATURE IMPORTANCE (RANDOM FOREST)
# =================================

rf_pipe = trained_pipelines["Random Forest"]
rf_model = rf_pipe.named_steps["model"]
rf_pre = rf_pipe.named_steps["preprocessor"]

# Get feature names after preprocessing
feature_names = rf_pre.get_feature_names_out()
rf_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

rf_importances.to_csv(TABLE_DIR / "table_random_forest_feature_importance.csv", index=False)

top_n = 20
plt.figure(figsize=(10, 7))
sns.barplot(
    data=rf_importances.head(top_n),
    x="importance",
    y="feature",
    palette="viridis"
)
plt.title(f"Top {top_n} Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_random_forest_feature_importance_top20.png", dpi=300)
plt.show()

rf_importances.head(10)


In [ ]:
# =========================
# 10. SHAP EXPLAINABILITY
# =========================

# SHAP with tree-based model is computationally efficient using TreeExplainer.
# To keep runtime manageable, we explain a sample from the test set.

sample_size = min(500, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)

# Transform sample using trained preprocessor
X_test_sample_transformed = rf_pre.transform(X_test_sample)

# Build SHAP explainer for Random Forest
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_sample_transformed)

# Handle binary classification output shape across SHAP versions
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_vals_for_class1 = shap_values[1]
else:
    shap_vals_for_class1 = shap_values

# Summary plot (bar)
plt.figure()
shap.summary_plot(
    shap_vals_for_class1,
    features=X_test_sample_transformed,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_summary_bar_top20.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary plot (beeswarm)
plt.figure()
shap.summary_plot(
    shap_vals_for_class1,
    features=X_test_sample_transformed,
    feature_names=feature_names,
    show=False,
    max_display=20,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_shap_summary_beeswarm_top20.png", dpi=300, bbox_inches="tight")
plt.show()


## Section G: Advanced Model Extensions

This section adds 2–3 stronger models often used in tabular classification:
1. Gradient Boosting Classifier
2. XGBoost (if installed)
3. (Optional) HistGradientBoosting as a fast alternative

### Why these models may improve performance
- Boosting methods combine weak learners sequentially to reduce bias.
- They often capture complex non-linear interactions better than linear models.
- They frequently perform strongly on structured business datasets.

### Dissertation writing guidance
- **Methodology:** Position these as advanced comparative benchmarks.
- **Results:** Compare against baseline models and discuss trade-off between performance and interpretability.


In [ ]:
# =====================================
# 11. ADVANCED MODELS IMPLEMENTATION
# =====================================

from sklearn.ensemble import HistGradientBoostingClassifier

advanced_models = {
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

if XGBOOST_AVAILABLE:
    # scale_pos_weight can be tuned when classes are imbalanced
    advanced_models["XGBoost"] = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

advanced_pipelines = {}
advanced_metrics = []

for name, model in advanced_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipe.fit(X_train, y_train)
    advanced_pipelines[name] = pipe

    m, cm, report, fpr, tpr = evaluate_model(name, pipe, X_test, y_test)
    advanced_metrics.append(m)
    print(f"\n{name} report:\n{report}")

advanced_metrics_df = pd.DataFrame(advanced_metrics).sort_values(by="F1", ascending=False)
advanced_metrics_df.to_csv(TABLE_DIR / "table_model_metrics_advanced.csv", index=False)
advanced_metrics_df


In [ ]:
# =============================================
# 12. COMBINED PERFORMANCE TABLE (ALL MODELS)
# =============================================

combined_metrics = pd.concat([metrics_df, advanced_metrics_df], axis=0, ignore_index=True)
combined_metrics = combined_metrics.sort_values(by="F1", ascending=False)
combined_metrics.to_csv(TABLE_DIR / "table_model_metrics_all.csv", index=False)

combined_metrics


## Section H: Ethical, Transparency, and Governance Reflection

### Suggested discussion points for dissertation
- The dataset is synthetic and anonymised, reducing direct privacy risk.
- Model predictions can still introduce **indirect bias** if feature patterns reflect structural inequalities.
- Explainability methods (e.g., SHAP) support transparency and accountability.
- Churn interventions should be monitored to avoid unfair profiling of customer groups.
- Governance should include:
  - periodic model audits,
  - performance monitoring over time,
  - documentation of model updates,
  - clear communication of model limitations.


## Section I: Dissertation Figure/Table Checklist

- **Figures generated by this notebook** (saved in `outputs/figures/`):
  - `fig_target_distribution.png`
  - `fig_numeric_distributions.png`
  - `fig_correlation_heatmap.png`
  - `fig_confusion_matrices_baseline.png`
  - `fig_roc_curves_baseline.png`
  - `fig_random_forest_feature_importance_top20.png`
  - `fig_shap_summary_bar_top20.png`
  - `fig_shap_summary_beeswarm_top20.png`

- **Tables generated by this notebook** (saved in `outputs/tables/`):
  - `table_missing_values.csv`
  - `table_model_metrics_baseline.csv`
  - `table_model_metrics_advanced.csv`
  - `table_model_metrics_all.csv`
  - `table_random_forest_feature_importance.csv`


## Section-by-Section Dissertation Notes (Quick Reference)

### 1) Data loading
- **What:** Import CSV, inspect dimensions/types.
- **Why:** Ensures reproducibility and data integrity.
- **Write-up placement:** Methodology (Data source and preparation).

### 2) EDA
- **What:** Summary statistics, class balance, distributions, correlations.
- **Why:** Identifies patterns and potential modelling challenges.
- **Write-up placement:** Results (Descriptive analysis).

### 3) Preprocessing
- **What:** Imputation, encoding, scaling in a reproducible pipeline.
- **Why:** Ensures proper treatment of mixed data types and prevents leakage.
- **Write-up placement:** Methodology (Preprocessing pipeline).

### 4) Model training
- **What:** Train Logistic Regression, Random Forest, SVM under common split.
- **Why:** Enables fair comparative benchmarking.
- **Write-up placement:** Methodology (Model development).

### 5) Evaluation
- **What:** Accuracy, precision, recall, F1, confusion matrix, ROC-AUC.
- **Why:** Captures multiple dimensions of performance, especially for churn class.
- **Write-up placement:** Results (Comparative performance).

### 6) Explainability (SHAP)
- **What:** Global feature influence and feature impact distribution.
- **Why:** Makes predictions transparent and supports ethical interpretation.
- **Write-up placement:** Results + Discussion (Explainability and trust).

### 7) Advanced models
- **What:** Add boosting-based models for stronger tabular performance.
- **Why:** Tests whether advanced learners improve predictive quality.
- **Write-up placement:** Methodology extension + Results comparison.
